In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


In [ ]:
# This is only needed on the software development machine. If you are a user, you can skip this step.
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
# Make sure you have pip installed neo4j
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Aperture import Aperture
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Plotly import Plotly
from topologicpy.Neo4j import Neo4j
from getpass import getpass
import uuid
import hashlib
import json

In [ ]:
def ByOffsetArea(face,
                area,
                offsetKey="offset",
                minOffsetKey="minOffset",
                maxOffsetKey="maxOffset",
                defaultMinOffset=0,
                defaultMaxOffset=1,
                maxIterations = 1,
                tolerance=0.0001,
                silent = False):
    """
    Creates an offset face from the input face based on the input area.

    Parameters
    ----------
    face : topologic_core.Face
        The input face.
    area : float
        The desired area of the created face.
    offsetKey : str , optional
        The edge dictionary key under which to store the offset value. The default is "offset".
    minOffsetKey : str , optional
        The edge dictionary key under which to find the desired minimum edge offset value. If a value cannot be found, the defaultMinOffset input parameter value is used instead. The default is "minOffset".
    maxOffsetKey : str , optional
        The edge dictionary key under which to find the desired maximum edge offset value. If a value cannot be found, the defaultMaxOffset input parameter value is used instead. The default is "maxOffset".
    defaultMinOffset : float , optional
        The desired minimum edge offset distance. The default is 0.
    defaultMaxOffset : float , optional
        The desired maximum edge offset distance. The default is 1.
    maxIterations: int , optional
        The desired maximum number of iterations to attempt to converge on a solution. The default is 1.
    tolerance : float , optional
        The desired tolerance. The default is 0.0001.
    silent : bool , optional
        If set to True, no error and warning messages are printed. Otherwise, they are. The default is False.
    
    Returns
    -------
    topologic_core.Face
        The created face.

    """
    from topologicpy.Wire import Wire
    from topologicpy.Face import Face
    from topologicpy.Topology import Topology
    from topologicpy.Dictionary import Dictionary
    import numpy as np
    from scipy.optimize import minimize

    def compute_offset_amounts(face,
                            area,
                            offsetKey="offset",
                            minOffsetKey="minOffset",
                            maxOffsetKey="maxOffset",
                            defaultMinOffset=0,
                            defaultMaxOffset=1,
                            maxIterations = 1,
                            tolerance=0.0001):
        
        initial_offsets = []
        bounds = []
        for edge in edges:
            d = _record_dictionary(edge)
            minOffset = Dictionary.ValueAtKey(d, minOffsetKey) or defaultMinOffset
            maxOffset = Dictionary.ValueAtKey(d, maxOffsetKey) or defaultMaxOffset
            # Initial guess: small negative offsets to shrink the polygon, within the constraints
            initial_offsets.append((minOffset + maxOffset) / 2)
            # Bounds based on the constraints for each edge
            bounds.append((minOffset, maxOffset))

        # Convert initial_offsets to np.array for efficiency
        initial_offsets = np.array(initial_offsets)
        iteration_count = [0]  # List to act as a mutable counter

        def objective_function(offsets):
            for i, edge in enumerate(edges):
                d = _record_dictionary(edge)
                d = Dictionary.SetValueAtKey(d, offsetKey, offsets[i])
                edge = Topology.SetDictionary(edge, d)
            
            # Offset the wire
            new_face = Face.ByOffset(face, offsetKey=offsetKey, silent=silent)
            # Check for an illegal wire. In that case, return a very large loss value.
            if not Topology.IsInstance(_as_topology(new_face), 'Face'):
                return (float("inf"))
            # Calculate the area of the new wire/face
            new_area = Face.Area(new_face)
            
            # The objective is the difference between the target hole area and the actual hole area
            # We want this difference to be as close to 0 as possible
            loss = (new_area - area) ** 2
            # If the loss is less than the tolerance, accept the result and return a loss of 0.
            if loss < tolerance:
                return 0
            # Otherwise, return the actual loss value.
            return loss 
        
        # Callback function to track and display iteration number
        def iteration_callback(xk):
            iteration_count[0] += 1  # Increment the counter
            if not silent:
                print(f"Face.ByOffsetArea - Information: Iteration {iteration_count[0]}")
        
        # Use scipy optimization/minimize to find the correct offsets, respecting the min/max bounds
        result = minimize(objective_function,
                        initial_offsets,
                        method = "Powell",
                        bounds=bounds,
                        options={ 'maxiter': maxIterations},
                        callback=iteration_callback
                        )

        # Return the offsets
        return result.x
    
    if not Topology.IsInstance(_as_topology(face), 'Face'):
        if not silent:
            print("Face.OffsetByArea - Error: The input face parameter is not a valid face. Returning None.")
        return None
    
    edges = Topology.Edges(_as_topology(face))
    # Compute the offset amounts
    offsets = compute_offset_amounts(face,
                            area = area,
                            offsetKey = offsetKey,
                            minOffsetKey = minOffsetKey,
                            maxOffsetKey = maxOffsetKey,
                            defaultMinOffset = defaultMinOffset,
                            defaultMaxOffset = defaultMaxOffset,
                            maxIterations = maxIterations,
                            tolerance = tolerance)
    # Set the edge dictionaries correctly according to the specified offsetKey
    for i, edge in enumerate(edges):
        d = _record_dictionary(edge)
        d = Dictionary.SetValueAtKey(d, offsetKey, offsets[i])
        edge = Topology.SetDictionary(edge, d)
            
    # Offset the face
    return_face = Face.ByOffset(face, offsetKey=offsetKey, silent=silent)
    if not Topology.IsInstance(_as_topology(face), 'Face'):
        if not silent:
            print("Face.OffsetByArea - Error: Could not create the offset face. Returning None.")
        return None
    return return_face

In [ ]:
# Create the face
wire = Wire.Einstein(radius=4.5)
hole_1 = Wire.Rectangle(origin=Topology.Centroid(_as_topology(wire)), width=2.5, length=1.25)
hole_1 = Wire.Fillet(_as_topology(hole_1), radius=0.4, sides=4)
hole_2 = Topology.Translate(_as_topology(hole_1), 0, -2.25, 0)
face = Face.ByWires(wire, [hole_1, hole_2])
Topology.Show(face)
edges = Topology.Edges(_as_topology(face))
for i, edge in enumerate(edges):
    if i%2 == 0:
        d = Dictionary.ByKeysValues(["minOffset", "maxOffset"], [0.05, 1])
    else:
        d = Dictionary.ByKeysValues(["minOffset", "maxOffset"], [0.05, 0.5])
    edge = Topology.SetDictionary(edge, d)

area = 50  # Desired hole area
print("Original Area:", Face.Area(face))
print("Desired Offset Area:", area)

offset_face = Face.ByOffsetArea(face,
                    area,
                    offsetKey="offset",
                    minOffsetKey="minOffset",
                    maxOffsetKey="maxOffset",
                    defaultMinOffset=0,
                    defaultMaxOffset=1,
                    maxIterations = 2,
                    tolerance=0.01,
                    silent = True,
                    numWorkers=1)

if Topology.IsInstance(_as_topology(offset_face), 'Face'):
    print("Hole Area:", Face.Area(offset_face))
    Topology.Show(edges, offset_face,camera=[0,0,3], up=[0,1,0])
else:
    print("Operation Failed!")